# Longor M3 launcher (`libero_10` task 0)

Runs the full Longor path (planner + executor + VLA service) on `libero_10` task 0.

Goal for M3:
- At least one successful end-to-end episode (`success=True` in run summary).

Notes:
- Uses current runtime default `VLA_ACTIONS_PER_CHUNK` from `src/config.py`.
- Section 7 streams full stdout/stderr live from a single run.


## 1 — Clone repo and capture secrets

In [ ]:
import getpass

github_token = getpass.getpass('GitHub PAT (repo scope): ')
openai_api_key = getpass.getpass('OpenAI API key: ')

In [ ]:
import os

if os.path.exists('/content/Longor'):
    os.chdir('/content/Longor')
    print('pulling repo...')
    !git pull
else:
    os.environ['CLONE_URL'] = f"https://{github_token}@github.com/pmish11/Longor-Robotics.git"
    !git clone $CLONE_URL "/content/Longor"
    os.chdir('/content/Longor')

## 2 — Headless rendering + app deps

In [ ]:
import os

os.environ['PYOPENGL_PLATFORM'] = 'egl'
os.environ['MUJOCO_GL'] = 'egl'

!apt-get update
!apt-get install -y cmake libegl-dev libgl1-mesa-dev libgles2-mesa-dev
%pip install egl_probe hf-egl-probe --no-build-isolation
%pip install -U pip
%pip install -r deps/requirements-libero.txt
%pip install -e .

# Silence robosuite macro recommendation warning for this runtime.
!python -m robosuite.scripts.setup_macros

## 3 — Pi0.5 (OpenPI) venv

In [ ]:
import os, subprocess, sys

VLA_PYTHON = '/content/Longor/.vla-venv/bin/python'
OPENPI_DIR = '/content/openpi'

def vla_pip(*args):
    subprocess.run([VLA_PYTHON, '-m', 'pip', *args], check=True)

%pip install --quiet virtualenv
if not os.path.exists('/content/Longor/.vla-venv'):
    subprocess.run([sys.executable, '-m', 'virtualenv', '/content/Longor/.vla-venv'], check=True)
vla_pip('install', '-U', 'pip')

if not os.path.isdir(OPENPI_DIR):
    subprocess.run([
        'git', 'clone', '--recurse-submodules',
        'https://github.com/Physical-Intelligence/openpi', OPENPI_DIR,
    ], check=True)
vla_pip('install', '-e', OPENPI_DIR)
vla_pip('install', '-r', 'deps/requirements-vla-pi05-linux.txt')
vla_pip('install', '-e', '.')

## 4 — Configure `.env` for Longor run

In [ ]:
with open('.env', 'w', encoding='utf-8') as f:
    f.write('\n'.join([
        f'OPENAI_API_KEY={openai_api_key}',
        'LOG_LEVEL=DEBUG',
        f'LLM_MODEL=gpt-4.1-mini',
        'SAVE_VIDEO=1',
    ]) + '\n')

print('.env written')

## 5 — Start VLA server + health

In [ ]:
import os, subprocess, time, urllib.request

VLA_LOG = '/tmp/longor_m3_vla_server.log'
vla_python = '/content/Longor/.vla-venv/bin/python'
openpi_root = '/content/openpi'
env = dict(os.environ)

# OpenPI sometimes isn't importable even after editable install; make its source visible.
extra_paths = [
    openpi_root,
    f'{openpi_root}/src',
    f'{openpi_root}/packages/openpi/src',
]
py_path = env.get('PYTHONPATH', '')
py_parts = [p for p in py_path.split(':') if p]
for path in extra_paths:
    if os.path.isdir(path) and path not in py_parts:
        py_parts.append(path)
env['PYTHONPATH'] = ':'.join(py_parts)

# Colab gsutil can fail on OpenPI composite objects when crcmod C extension is unavailable.
# For this notebook run, skip hash checks so checkpoint download can complete.
boto_cfg = '/tmp/longor_m3.boto'
with open(boto_cfg, 'w', encoding='utf-8') as f:
    f.write('[GSUtil]\ncheck_hashes = never\n')
env['BOTO_CONFIG'] = boto_cfg

check = subprocess.run(
    [vla_python, '-c', 'import openpi'],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
if check.returncode != 0:
    print('OpenPI import failed in VLA venv; retrying editable install...')
    reinstall = subprocess.run(
        [vla_python, '-m', 'pip', 'install', '-e', openpi_root],
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if reinstall.returncode != 0:
        print(reinstall.stdout)
        raise RuntimeError('Failed reinstalling OpenPI editable package in .vla-venv')

    check = subprocess.run(
        [vla_python, '-c', 'import openpi'],
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if check.returncode != 0:
        print(check.stdout)
        raise RuntimeError('OpenPI still not importable in .vla-venv')

log = open(VLA_LOG, 'w', buffering=1)
proc = subprocess.Popen(
    [vla_python, 'services/run_vla_server.py'],
    cwd='/content/Longor',
    stdout=log,
    stderr=subprocess.STDOUT,
    env=env,
)

health_ok = False
max_wait_s = 1800
for elapsed in range(0, max_wait_s, 5):
    if proc.poll() is not None:
        raise RuntimeError(f'VLA server exited early with code {proc.returncode}; see {VLA_LOG}')
    try:
        resp = urllib.request.urlopen('http://localhost:8765/health', timeout=5)
        print('Health:', resp.read().decode())
        health_ok = True
        break
    except Exception:
        print(f'Waiting for /health ({elapsed}s)...')
        time.sleep(5)

if not health_ok:
    raise RuntimeError(f'VLA server not healthy after {max_wait_s}s; see {VLA_LOG}')

## 6 — Run Longor once (`libero_10`, task 0)


In [ ]:
!python examples/long_horizon_llm_libero.py

## 7 — Save the video

### HOW TO GET VIDEO FILES (when running via Colab extension)?
1. run with 'SAVE_VIDEO=1'
2. click Colab -> mount server to workspace
3. reload IDE window.
4. open the files sidebar! :)
